# 1.对话消息历史组件基础

知识点讲解：ChatMessageHistory 对话消息历史组件

1. 记忆功能的两个核心问题与设计动机
   - 把记忆功能集成到 LLM 应用，涉及两个核心问题：存储的历史信息是什么？如何检索与处理历史信息？
   - 一个记忆类由「历史消息的存储」与「记忆的处理逻辑」两部分组成，LangChain 采用同样的思路
   - 朴素写法里「检索历史」和「存储历史」逻辑常混在一起；LangChain 用专门组件把「存储」职责单独抽离，便于替换与测试

2. 什么是 BaseChatMessageHistory
   - LangChain 封装的管理历史信息的抽象基类，所有扩展的消息历史组件（含自定义组件）均继承它
   - 职责单一：只管消息的增（add_*）、查（messages）、删（clear），不参与 Prompt 拼装
   - 在 LangChain 1.x 中，它取代了旧的 Memory 体系，成为记忆能力的唯一存储底座

3. BaseChatMessageHistory 的核心接口
   - messages: List[BaseMessage]        属性，读取全部历史消息
   - add_user_message / add_ai_message   追加人类 / AI 消息
   - add_message(message)                追加任意类型的单条消息
   - add_messages(messages)              批量追加消息列表
   - clear()                             清空全部历史
   - aget_messages / aadd_messages       对应的异步版本

4. 消息类型体系（langchain_core.messages）
   - SystemMessage  系统指令，设定 AI 角色与行为准则
   - HumanMessage   人类用户输入
   - AIMessage      AI 回复，可携带 tool_calls
   - ToolMessage    工具执行结果，需与 tool_call_id 对应
   - FunctionMessage 旧版函数调用结果（已被 ToolMessage 取代）
   - 所有类型都继承自 BaseMessage，含 content、additional_kwargs、response_metadata

5. InMemoryChatMessageHistory 的特点
   - 位于 langchain_core.chat_history，是最轻量的内存实现，也是 langchain_core 内置的对话消息历史类
   - 底层就是一个 Python list，进程退出后数据即丢失
   - 适用：单元测试、教学演示、单次会话的临时缓存
   - 不适用：多进程/多实例部署、需要持久化的生产环境

6. 常见的持久化实现（langchain_community.chat_message_histories）
   - 第三方集成的记忆组件均通过 langchain_community 导入（与内置的 InMemory 不同）
   - FileChatMessageHistory        存到本地 JSON 文件（见本章第 2 个示例）
   - RedisChatMessageHistory       存到 Redis，支持 TTL 过期，生产最常用
   - SQLChatMessageHistory / PostgresChatMessageHistory / MongoDBChatMessageHistory  存到各类数据库
   - 它们的 API 完全一致，切换存储只需替换构造函数，业务代码无需改动

7. 与 Memory 组件的区别（版本演进）
   - LangChain 0.x：Memory 既存历史，又隐式改写链的输入输出，行为不透明
   - LangChain 1.x：ChatMessageHistory 只存历史，注入 Prompt 由 MessagesPlaceholder + RunnableWithMessageHistory 显式完成
   - 好处：职责清晰、行为可预测、便于测试与替换存储后端

8. 典型使用方式
   - 手动注入：chain.invoke({"query": q, "history": chat_history.messages})
   - 自动读写：用 RunnableWithMessageHistory 包装链，按 session_id 自动存取

9. 注意事项
   - messages 返回的是消息对象列表，不是字符串；转字符串用 get_buffer_string(messages)
   - 该组件本身不做任何裁剪，历史会无限增长，需配合 trim_messages 控制 token
   - InMemoryChatMessageHistory 非线程安全，并发写入需自行加锁


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message("你好，我是慕小课，你是谁？")
chat_history.add_ai_message("你好，我是DeepSeek，有什么可以帮到您的？")
print(chat_history.messages)